bibliotecas

In [1]:
import pandas as pd
import kagglehub
import os


c:\Users\danil\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


data set

In [2]:
path = kagglehub.dataset_download("olistbr/brazilian-ecommerce")

print("Path to dataset files:", path)
print()
print(os.listdir(path))

Path to dataset files: C:\Users\danil\.cache\kagglehub\datasets\olistbr\brazilian-ecommerce\versions\2

['olist_customers_dataset.csv', 'olist_geolocation_dataset.csv', 'olist_orders_dataset.csv', 'olist_order_items_dataset.csv', 'olist_order_payments_dataset.csv', 'olist_order_reviews_dataset.csv', 'olist_products_dataset.csv', 'olist_sellers_dataset.csv', 'product_category_name_translation.csv']


In [20]:
customers = pd.read_csv(os.path.join(path, "olist_customers_dataset.csv")) # Cliente
orders = pd.read_csv(os.path.join(path, "olist_orders_dataset.csv")) # Pedidos
order_items = pd.read_csv(os.path.join(path, "olist_order_items_dataset.csv")) # Pedidos - Items
payments = pd.read_csv(os.path.join(path, "olist_order_payments_dataset.csv")) # Pagamentos
reviews = pd.read_csv(os.path.join(path, "olist_order_reviews_dataset.csv")) # NPS
products = pd.read_csv(os.path.join(path, "olist_products_dataset.csv")) # Produtos
sellers = pd.read_csv(os.path.join(path, "olist_sellers_dataset.csv")) # Vendedores/Lojas
geolocation = pd.read_csv(os.path.join(path, "olist_geolocation_dataset.csv")) # Geo
category_translation = pd.read_csv(os.path.join(path, "product_category_name_translation.csv")) # Traducao produtos.

tratando data set

In [26]:
cols_orders = [
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date"
]

cols_reviews = ["review_creation_date", "review_answer_timestamp"]

order_items["shipping_limit_date"] = pd.to_datetime(order_items["shipping_limit_date"]) # .dt.date

for col in cols_orders:
    orders[col] = pd.to_datetime(orders[col]) # .dt.date

for col in cols_reviews:
    reviews[col] = pd.to_datetime(reviews[col]) # .dt.date

criando colunas essenciais (gráficos e análises)

In [ ]:
# compras por ano-mês 
orders["purchase_year_month"] = orders["order_purchase_timestamp"].dt.to_period("M")

# compras por ano e mês (separados)
orders["purchase_year"] = orders["order_purchase_timestamp"].dt.year
orders["purchase_month"] = orders["order_purchase_timestamp"].dt.month

# qtd dias para o cliente receber. (data de entrega menos data da compra)
orders["delivery_days"] = (orders["order_delivered_customer_date"] - orders["order_purchase_timestamp"]).dt.days

# qtd dias para aprovar pedido/pagamento (data da aprovação menos a data da compra)
orders["approval_days"] = (orders["order_approved_at"] - orders["order_purchase_timestamp"]).dt.days

# qtd dias para postar na transportadora (data de entrega na transportadora menos data da aprovação)
orders["carrier_days"] = (orders["order_delivered_carrier_date"] - orders["order_approved_at"]).dt.days

# prazo prometido ao cliente  (data esperada da entrega menos data da compra.)
orders["estimated_delivery_days"] = (orders["order_estimated_delivery_date"] - orders["order_purchase_timestamp"]).dt.days

# calculando atraso (data de entrega menos data estimada da entrega)
orders["delay_days"] = (orders["order_delivered_customer_date"] - orders["order_estimated_delivery_date"]).dt.days

# flag de atraso (se o delay for maior do que zero, que dizer que atrasou, se não, não)
orders["is_late"] = (orders["delay_days"] > 0).astype(int)

In [34]:
# valor total do pedido (preço + frete)

order_items["item_total"] = order_items["price"] + order_items["freight_value"]

In [ ]:
# conferindo se tem mais de um pagamento por order_id

# payments["order_id"].value_counts()

order_id
fa65dad1b0e818e3ccc5cb0e39231352    29
ccf804e764ed5650cd8759557269dc13    26
285c2e15bebd4ac83635ccc563dc71f4    22
895ab968e7bb0d5659d16cd74cd1650c    21
ee9ca989fc93ba09a6eddc250ce01742    19
                                    ..
0406037ad97740d563a178ecc7a2075c     1
7b905861d7c825891d6347454ea7863f     1
32609bbb3dd69b3c066a6860554a77bf     1
b8b61059626efa996a60be9bb9320e10     1
28bbae6599b09d39ca406b747b6632b1     1
Name: count, Length: 99440, dtype: int64

In [60]:
reviews["review_is_good"] = (reviews["review_score"] >= 4).astype(int)

reviews["review_is_bad"] = (reviews["review_score"] <= 2).astype(int)

reviews["review_response_days"] = (
    reviews["review_answer_timestamp"] - reviews["review_creation_date"]
).dt.days

agregando (group by)

In [62]:
order_items_summary = order_items.groupby("order_id").agg(
    items_count=("order_item_id", "count"),
    items_total_price=("price", "sum"),
    items_total_freight=("freight_value", "sum"),
    items_total_value=("item_total", "sum")
).reset_index()

order_items_summary.head()

,order_id,items_count,items_total_price,items_total_freight,items_total_value
0,00010242fe8c5a6d1ba2dd792cb16214,1,58.90,13.29,72.19
1,00018f77f2f0320c557190d7a144bdd3,1,239.90,19.93,259.83
2,000229ec398224ef6ca0657da4fc703e,1,199.00,17.87,216.87
3,00024acbcdf0a6daa1e931b038114c75,1,12.99,12.79,25.78
4,00042b26cf59d7ce69dfabb4e55b4fd9,1,199.90,18.14,218.04


In [63]:
reviews_summary = reviews.groupby("order_id").agg(
    review_score_mean=("review_score", "mean"),
    review_is_good_rate=("review_is_good", "mean"),
    review_is_bad_rate=("review_is_bad", "mean"),
    review_response_days_mean=("review_response_days", "mean")
).reset_index()

reviews_summary.head()

,order_id,review_score_mean,review_is_good_rate,review_is_bad_rate,review_response_days_mean
0,00010242fe8c5a6d1ba2dd792cb16214,5.0,1.0,0.0,1.0
1,00018f77f2f0320c557190d7a144bdd3,4.0,1.0,0.0,2.0
2,000229ec398224ef6ca0657da4fc703e,5.0,1.0,0.0,0.0
3,00024acbcdf0a6daa1e931b038114c75,4.0,1.0,0.0,0.0
4,00042b26cf59d7ce69dfabb4e55b4fd9,5.0,1.0,0.0,1.0


In [64]:
# agrupando os pedidos e calculando o valor total de pagamento, média de parcelas e a principal forma de pagamento.

payments_summary = payments.groupby("order_id").agg(
    payment_value_total=("payment_value", "sum"),
    installments_mean=("payment_installments", "mean"),
    payment_type_main=("payment_type", "first")
).reset_index()

payments_summary.head()

,order_id,payment_value_total,installments_mean,payment_type_main
0,00010242fe8c5a6d1ba2dd792cb16214,72.19,2.0,credit_card
1,00018f77f2f0320c557190d7a144bdd3,259.83,3.0,credit_card
2,000229ec398224ef6ca0657da4fc703e,216.87,5.0,credit_card
3,00024acbcdf0a6daa1e931b038114c75,25.78,2.0,credit_card
4,00042b26cf59d7ce69dfabb4e55b4fd9,218.04,3.0,credit_card


join

In [65]:
fact_orders = (
    orders
    .merge(customers, on="customer_id", how="left")
    .merge(order_items_summary, on="order_id", how="left")
    .merge(payments_summary, on="order_id", how="left")
    .merge(reviews_summary, on="order_id", how="left")
)

In [66]:
fact_orders.shape[0] == fact_orders["order_id"].nunique()

True

In [69]:
fact_orders.describe()

,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,purchase_year,purchase_month,delivery_days,approval_days,carrier_days,...,items_count,items_total_price,items_total_freight,items_total_value,payment_value_total,installments_mean,review_score_mean,review_is_good_rate,review_is_bad_rate,review_response_days_mean
count,99441,99281,97658,96476,99441,99441.000000,99441.000000,96476.000000,99281.000000,97644.000000,...,98666.000000,98666.000000,98666.000000,98666.000000,99440.000000,99440.000000,98673.000000,98673.000000,98673.000000,98673.000000
mean,2017-12-30 17:26:56,2017-12-31 05:20:33,2018-01-04 06:13:53,2018-01-13 19:24:32,2018-01-24 03:08:37,2017.539838,6.032220,12.497336,0.518508,2.707191,...,1.141731,137.754076,22.823562,160.577638,160.990267,2.914696,4.086793,0.770839,0.146798,2.582687
min,2016-09-04 00:00:00,2016-09-15 00:00:00,2016-10-08 00:00:00,2016-10-11 00:00:00,2016-09-30 00:00:00,2016.000000,1.000000,0.000000,0.000000,-171.000000,...,1.000000,0.850000,0.000000,9.590000,0.000000,0.000000,1.000000,0.000000,0.000000,0.000000
25%,2017-09-12 00:00:00,2017-09-12 00:00:00,2017-09-15 00:00:00,2017-09-25 00:00:00,2017-10-03 00:00:00,2017.000000,3.000000,7.000000,0.000000,1.000000,...,1.000000,45.900000,13.850000,61.980000,62.010000,1.000000,4.000000,1.000000,0.000000,1.000000
50%,2018-01-18 00:00:00,2018-01-19 00:00:00,2018-01-24 00:00:00,2018-02-02 00:00:00,2018-02-15 00:00:00,2018.000000,6.000000,10.000000,0.000000,2.000000,...,1.000000,86.900000,17.170000,105.290000,105.290000,2.000000,5.000000,1.000000,0.000000,1.000000
75%,2018-05-04 00:00:00,2018-05-04 00:00:00,2018-05-08 00:00:00,2018-05-15 00:00:00,2018-05-25 00:00:00,2018.000000,8.000000,16.000000,1.000000,3.000000,...,1.000000,149.900000,24.040000,176.870000,176.970000,4.000000,5.000000,1.000000,0.000000,3.000000
max,2018-10-17 00:00:00,2018-09-03 00:00:00,2018-09-11 00:00:00,2018-10-17 00:00:00,2018-11-12 00:00:00,2018.000000,12.000000,210.000000,188.000000,126.000000,...,21.000000,13440.000000,1794.960000,13664.080000,13664.080000,24.000000,5.000000,1.000000,1.000000,518.000000
std,NaN,NaN,NaN,NaN,NaN,0.505007,3.232999,9.555460,1.171324,3.568131,...,0.538452,210.645145,21.650909,220.466087,221.951257,2.700263,1.346274,0.419940,0.353562,9.898589


a partir daqui é gráficos e histórias que queremos contar.


subir isso no github - fazer gráficos no power bi?